In [1]:
# Import necessary packages
import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

In [3]:
# Function 1: Inspect all input rasters
def inspect_rasters(raster_dir, feature_names):
    """Inspect each raster's basic properties: CRS, resolution, min, max, mean, NoData."""
    print("\n Inspecting Input Raster Files:\n")
    
    for feature in feature_names:
        raster_path = os.path.join(raster_dir, f"{feature}.tif")
        try:
            with rasterio.open(raster_path) as src:
                data = src.read(1)

                print(f"Raster: {feature}")
                print(f" - CRS: {src.crs}")
                print(f" - Resolution: {src.res}")
                print(f" - Shape: {src.width} x {src.height}")
                print(f" - Min Value: {np.nanmin(data):.2f}")
                print(f" - Max Value: {np.nanmax(data):.2f}")
                print(f" - Mean Value: {np.nanmean(data):.2f}")
                print(f" - NoData Value: {src.nodata}")
                print("-"*50)
        
        except Exception as e:
            print(f" Could not read {feature}.tif: {e}")
            continue

#  Function 2: Align rasters
def align_rasters(raster_paths, reference_meta):
    """Align all rasters to the same resolution and size."""
    aligned_rasters = []
    for raster_path in raster_paths:
        with rasterio.open(raster_path) as src:
            data = src.read(
                1, out_shape=(reference_meta['height'], reference_meta['width']),
                resampling=Resampling.bilinear
            )
            aligned_rasters.append(data)
    return np.array(aligned_rasters)

#  Function 3: Predict spatial yield map
def predict_spatial_yield(raster_dir, model, scaler, feature_names, output_path):
    """Predict yield spatially across all raster pixels."""
    raster_paths = [os.path.join(raster_dir, f"{f}.tif") for f in feature_names]

    # Inspect rasters first
    inspect_rasters(raster_dir, feature_names)

    # Load reference metadata
    with rasterio.open(raster_paths[0]) as src:
        reference_meta = src.meta
        reference_meta.update(dtype=rasterio.float32, count=1)

    # Align all rasters
    aligned_rasters = align_rasters(raster_paths, reference_meta)
    height, width = aligned_rasters.shape[1:]
    raster_stack_flat = aligned_rasters.reshape(aligned_rasters.shape[0], -1).T

    # Cleaning step
    raster_stack_flat = np.nan_to_num(raster_stack_flat, nan=0.0, posinf=0.0, neginf=0.0)
    raster_stack_flat = np.clip(raster_stack_flat, -1e3, 1e3)

    # Print raster stack summary
    print("\n Raster Stack Summary Before Scaling and Prediction:")
    print(f"Minimum: {np.min(raster_stack_flat):.2f}")
    print(f"Maximum: {np.max(raster_stack_flat):.2f}")
    print(f"Mean: {np.mean(raster_stack_flat):.2f}")
    print(f"Standard Deviation: {np.std(raster_stack_flat):.2f}")

    # Normalize using same scaler
    raster_stack_flat = scaler.transform(raster_stack_flat)

    # Predict
    yield_prediction_flat = model.predict(raster_stack_flat)
    yield_prediction = yield_prediction_flat.reshape(height, width)

    # Save the predicted map
    with rasterio.open(output_path, "w", **reference_meta) as dest:
        dest.write(yield_prediction.astype(rasterio.float32), 1)

    print(f"\n Predicted spatial yield map saved at: {output_path}")



# Define paths
raster_dir = r"..\..\Crop Predicion\DATA"  # Folder containing input rasters
output_yield_tiff = r"..\..\Crop Predicion\DATA\spatial_yield_prediction3.tif"  #  Output prediction path

# Define feature names (raster names without .tif)
feature_names = ['ndvi', 'NDRE', 'EVI', 'SAVI1', 'fapar', 'LAI', 'VH', 'VV', 'entropy', 'alpha', 'rvi', 'rfcum', 'humidity', 'solarrad', 'gddcumulative', 'cc', 'nitro', 'pH', 'OC']


# Load your CSV (already processed)
df = pd.read_csv(r"..\..\Crop Predicion\DATA\extracted3004.csv")  # Your training data

# Prepare X and y
X = df[feature_names]
y = df['yield']

# Split into train-test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalize
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Random Forest
rf_model = RandomForestRegressor(random_state=42, n_estimators=100)
rf_model.fit(X_train_scaled, y_train)

# Final spatial yield prediction
predict_spatial_yield(raster_dir, rf_model, scaler, feature_names, output_yield_tiff)

import pickle

# Save model
with open("yield_model_oj.pkl", "wb") as f:
    pickle.dump(rf_model, f)

# Save scaler
with open("yield_scaler_oj.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Model and scaler saved successfully!")



 Inspecting Input Raster Files:

Raster: ndvi
 - CRS: EPSG:32644
 - Resolution: (10.0, 10.0)
 - Shape: 2210 x 3356
 - Min Value: -340282346638528859811704183484516925440.00
 - Max Value: 0.73
 - Mean Value: -inf
 - NoData Value: -3.4028234663852886e+38
--------------------------------------------------
Raster: NDRE
 - CRS: EPSG:32644
 - Resolution: (10.0, 10.0)
 - Shape: 2210 x 3356
 - Min Value: -340282346638528859811704183484516925440.00
 - Max Value: 0.81


C:\Users\neela\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\numpy\_core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 - Mean Value: -inf
 - NoData Value: -3.4028234663852886e+38
--------------------------------------------------
Raster: EVI
 - CRS: EPSG:32644
 - Resolution: (10.0, 10.0)
 - Shape: 2210 x 3356
 - Min Value: -340282346638528859811704183484516925440.00
 - Max Value: 1.00
 - Mean Value: -inf
 - NoData Value: -3.4028234663852886e+38
--------------------------------------------------
Raster: SAVI1
 - CRS: EPSG:32644
 - Resolution: (10.0, 10.0)
 - Shape: 2210 x 3356
 - Min Value: -179769313486231570814527423731704356798070567525844996598917476803157260780028538760589558632766878171540458953514382464234321326889464182768467546703537516986049910576551282076245490090389328944075868508455133942304583236903222948165808559332123348274797826204144723168738177180919299881250404026184124858368.00
 - Max Value: 1.50
 - Mean Value: -inf
 - NoData Value: -1.7976931348623157e+308
--------------------------------------------------
Raster: fapar
 - CRS: EPSG:32644
 - Resolution: (10.0, 10.0)
 - Shape: 2210

C:\Users\neela\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(



 Predicted spatial yield map saved at: ..\..\Crop Predicion\DATA\spatial_yield_prediction3.tif
Model and scaler saved successfully!
